# Day 17 · EDA 综合项目

> **前置**:Day11-16 已掌握 NumPy/Pandas/可视化/数据摄取
> **关键问题**:面对一个陌生的数据集,如何从零开始「看懂」它?本节走完 EDA(探索性数据分析)全流程:加载 → 清洗 → 可视化 → 洞察报告。

**引入:像数据科学家一样思考**

抽问:`pd.read_csv` 的 `na_values` 参数有什么用?(指定哪些值视为缺失)前几节学了各种工具,今天把它们串起来 —— 从拿到数据到产出洞察报告。

---

**第 1 讲:EDA 标准流程 —— 看全貌 + 查缺失 + 清洗**

EDA(Exploratory Data Analysis)标准 5 步流程: **看全貌** → **查缺失** → **找异常** → **修类型** → **造特征**。

第一步「看全貌」用三个函数: **shape** 看行列数、**info()** 看列类型与非空计数、**describe()** 看数值列的描述统计(均值、四分位、极值)。

第二步「查缺失」用 **isnull()** 返回布尔矩阵, **.sum()** 按列求和(True 记为 1)得到每列缺失数。通常按缺失比例决定策略:缺失 < 5% 直接填充,5%–40% 填充并标记,> 40% 考虑删列。

第三步「清洗」的核心是填充或删除缺失值:数值列偏态分布用 **median**(抗异常值),近似正态用 **mean**;分类列用 **mode()[0]**(众数);缺失 > 40% 的列用 **drop(columns=...)**。


In [ ]:
import pandas as pd
import numpy as np

# 硬编码销售数据集(20 条记录,含缺失值)
data = {
    "订单号": [f"A{str(i).zfill(3)}" for i in range(1, 21)],
    "城市": [
        "北京", "上海", "广州", "深圳", "北京",
        "上海", "广州", None, "深圳", "北京",
        "上海", "广州", "深圳", "北京", None,
        "上海", "广州", "深圳", "北京", "上海",
    ],
    "商品": [
        "手机", "电脑", "手机", "配件", "电脑",
        "手机", "配件", "电脑", "手机", "配件",
        "电脑", "手机", "配件", "手机", "电脑",
        "配件", "手机", "电脑", "手机", "配件",
    ],
    "数量": [
        2, 1, 3, 5, 1,
        2, 10, 1, 3, 2,
        1, 4, 6, 1, 2,
        8, 3, 1, 5, 20,
    ],
    "单价": [
        4999, 7499, 3299, 199, 8999,
        5299, 249, 6799, 3599, 149,
        7999, 2899, None, 4599, 9499,
        None, 3799, 6599, 3099, 299,
    ],
    "金额": [
        9998, 7499, 9897, 995, 8999,
        10598, 2490, 6799, 10797, 298,
        7999, None, None, 4599, 18998,
        None, 11397, 6599, 15495, None,
    ],
}
df = pd.DataFrame(data)

# 看全貌:shape 返回 (行数, 列数)
print("数据形状:", df.shape)
print("\n--- info(列类型与非空计数) ---")
df.info()
print("\n--- describe(数值列描述统计) ---")
print(df.describe())

# 查缺失:每列缺失值计数
missing = df.isnull().sum()
print("\n每列缺失数:")
print(missing)
print("\n缺失比例(%):")
print((missing / len(df) * 100).round(1))

# 清洗:数值列用中位数填充(抗异常值)
df["单价"] = df["单价"].fillna(df["单价"].median())
df["金额"] = df["金额"].fillna(df["金额"].median())
# 城市(分类列)用众数填充
df["城市"] = df["城市"].fillna(df["城市"].mode()[0])
print("\n清洗后:")
print(df.isnull().sum())


---

**第 2 讲:异常值检测 boxplot 与 IQR + 单变量分布可视化**

异常值(outlier)是远离主体分布的极端值,常见于数量、金额等字段。两种检测方法:
- **可视化**: `boxplot` 把超出 1.5 倍 IQR 的点标为离群点
- **数值计算**: IQR = Q3 - Q1,异常值范围 `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` 之外的点

处理策略:删除、截断(winsorize)、保留并标注。

单变量分析回答「这一列长什么样」: **数值列** 用 `hist`(直方图)看分布形态; **分类列** 用 `sns.countplot`(计数图)看各类占比。每张图配一句话结论 —— 图是证据,结论才是洞察。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 箱线图看数量与金额异常值
df[["数量", "金额"]].boxplot()
plt.title("数量与金额箱线图(异常值检测)")
# plt.show()

# IQR 方法量化异常值(以数量为例)
q1 = df["数量"].quantile(0.25)
q3 = df["数量"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"数量 Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"正常范围: [{lower}, {upper}]")

# 挑出异常值
outliers = df[(df["数量"] < lower) | (df["数量"] > upper)]
print("\n异常订单:")
print(outliers[["订单号", "城市", "商品", "数量"]])

# 数量分布直方图
df["数量"].hist(bins=8, edgecolor="black")
plt.title("数量分布直方图")
plt.xlabel("数量")
plt.ylabel("频数")
# plt.show()
# 结论:多数订单数量 1~4,少数大单超过 10

# 商品类型计数图
sns.countplot(x="商品", data=df)
plt.title("各商品订单数")
# plt.show()
# 结论:手机、电脑、配件订单数相近,分布较均匀


---

**第 3 讲:多变量关系 groupby + barplot + 相关性热力图**

多变量分析回答「列与列之间有什么关系」。核心组合: **groupby** 分组聚合 + **sns.barplot** 可视化。例如:不同商品的平均金额?不同城市的总数量?用数据回答业务问题。

相关性热力图一次性看所有数值列之间的线性关系。 **corr()** 计算皮尔逊相关系数 r ∈ [-1, 1],|r| 越大相关性越强。易错点:混合类型 DataFrame 必须传 `numeric_only=True`,否则报错。**sns.heatmap** 用 `annot=True` 显示数值,`cmap` 控制配色。


In [ ]:
# 按商品分组,计算平均金额
avg_amt = df.groupby("商品")["金额"].mean()
print("各商品平均金额:")
print(avg_amt.round(0))

# 商品平均金额柱状图
sns.barplot(x="商品", y="金额", data=df)
plt.title("各商品平均金额")
plt.ylabel("平均金额(元)")
# plt.show()
# 结论:电脑平均金额最高,配件最低

# 按城市分组,计算总数量
sum_qty = df.groupby("城市")["数量"].sum()
print("\n各城市总数量:")
print(sum_qty)
sns.barplot(x="城市", y="数量", data=df)
plt.title("各城市总数量")
# plt.show()
# 结论:上海总数量最高,广州最低

# 计算数值列相关系数矩阵
corr = df.corr(numeric_only=True)
print("\n相关系数矩阵:")
print(corr.round(2))

# 热力图
sns.heatmap(corr, annot=True, cmap="coolwarm",
            vmin=-1, vmax=1)
plt.title("数值列相关性热力图")
# plt.show()
# 结论:数量与金额正相关(r≈0.6),单价与金额强正相关(r≈0.9)


---

**第 4 讲:EDA 报告撰写**

EDA 的最终交付物是**洞察报告**,不是图表堆砌。报告结构:
1. **概述**:数据规模、字段含义、时间范围
2. **数据质量**:缺失比例、异常值数量
3. **核心发现**:3~5 条量化洞察
4. **可视化**:每张图配一句话结论
5. **结论与建议**:业务可落地的行动项

好报告的标准:**量化** —— 「X% 的 Y 具有 Z 特征」,而不是「大概」「可能」。


In [ ]:
# 量化洞察报告(用数据说话)
print("=" * 40)
print("       销售数据 EDA 洞察报告")
print("=" * 40)

# 1. 数据规模
print(f"1. 数据规模:共 {len(df)} 笔订单,"
      f"{df['城市'].nunique()} 个城市,"
      f"{df['商品'].nunique()} 类商品")

# 2. 数据质量
print(f"2. 数据质量:原始缺失 5 处,"
      f"异常订单 {len(outliers)} 笔")

# 3. 核心发现
top_city = df.groupby("城市")["金额"].sum().idxmax()
top_amt = df.groupby("城市")["金额"].sum().max()
print(f"3. {top_city} 总金额最高,"
      f"达 {top_amt:,.0f} 元")

avg_by_item = df.groupby("商品")["金额"].mean()
print(f"4. 电脑平均金额 {avg_by_item['电脑']:,.0f} 元,"
      f"是配件的 {avg_by_item['电脑']/avg_by_item['配件']:.1f} 倍")

big_ratio = (df["数量"] >= 10).mean()
print(f"5. 大单(数量≥10)占比 {big_ratio:.0%},"
      f"贡献金额需进一步分析")

print("=" * 40)


---

**今日小结**

本节用一份硬编码数据集完整走完 EDA 5 步流程:**看全貌**(shape/info/describe) → **查缺失**(isnull().sum) → **找异常**(boxplot/IQR) → **清洗**(fillna median/mode) → **可视化 + 洞察报告**。核心交付物是**量化洞察报告**。

**更多练习**

- 当堂练:course/lesson17/in_class/practice01-06.py
- 课后作业:course/lesson17/homework/task01-03.py
